### Task 3: Data Preprocessing
- Label data for speaker changes (binary labels: 1 for change, 0 for no change). 
- Do normalization
- Divide the data into train (session 1-3), validation (session 4), and test (session 5) sets.

## Experiment 1
**frame_length/sliding window:** 20ms (default)

**frame_shift/overlap:** 10ms (default)

In [4]:
import os
import pandas as pd
from sklearn.preprocessing import StandardScaler

def read_rttm_file(rttm_path):
    """
    Read RTTM file and return a list of speaker segments
    """
    speaker_segments = []
    with open(rttm_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if parts[0] == 'SPEAKER':
                speaker_segments.append({
                    'file': parts[1],
                    'time_start': float(parts[3]),
                    'duration': float(parts[4]),
                    'end_time': float(parts[3]) + float(parts[4]),
                })
    return speaker_segments

def label_speaker_changes(features_df, base_path):
    """
    Label speaker changes based on matching RTTM files
    """
    # Create a copy of the DataFrame to avoid modifying the original
    labeled_df = features_df.copy()
    
    # Add speaker change label column
    labeled_df['speaker_change'] = 0
    
    # Find unique wav files
    unique_wav_files = features_df['file_name'].unique()
    
    for wav_file in unique_wav_files:
        # Remove .wav extension and find matching RTTM
        base_filename = os.path.splitext(wav_file)[0]
        
        # Search for RTTM file in the entire base path
        matching_rttm_files = []
        for root, dirs, files in os.walk(base_path):
            matching_rttm_files.extend([
                os.path.join(root, f) for f in files 
                if base_filename in f and f.endswith('.rttm')
            ])
        
        if not matching_rttm_files:
            print(f"No RTTM file found for {wav_file}")
            continue
        
        # Use the first matching RTTM file
        rttm_path = matching_rttm_files[0]
        
        # Read speaker segments from RTTM file
        speaker_segments = read_rttm_file(rttm_path)
        
        # Label speaker changes
        for segment in speaker_segments:
            # Find rows within the speaker segment
            mask = (
               (labeled_df['file_name'] == wav_file) & 
               (labeled_df['start_time'] < segment['time_start']) & 
               (labeled_df['end_time'] >= segment['time_start'])
           )
            
            labeled_df.loc[mask, 'speaker_change'] = 1
    
    return labeled_df

def normalize_features_zscore(labeled_df):
    """
    Normalize numerical features using Z-score normalization
    """
    # Select numerical feature columns
    feature_columns = [col for col in labeled_df.columns if col.startswith('mfcc') or col == 'f0']
    
    # Initialize StandardScaler
    scaler = StandardScaler()
    
    # Apply Z-score normalization
    labeled_df[feature_columns] = scaler.fit_transform(labeled_df[feature_columns])
    
    print("Z-score normalization applied to numerical features.")
    return labeled_df

def process_all_sessions(base_path):
    """
    Process all sessions, label speaker changes, and normalize features
    """
    # Find feature CSV files
    feature_files = [f for f in os.listdir('.') if f.startswith('features1_') and f.endswith('.csv')]
    
    for features_csv in feature_files:
        # Load features
        features_df = pd.read_csv(features_csv)
        
        # Label speaker changes
        labeled_df = label_speaker_changes(features_df, base_path)
        
        # Normalize numerical features
        labeled_df = normalize_features_zscore(labeled_df)
        
        # Save labeled and normalized features
        output_csv = f'labeled_{features_csv}'
        labeled_df.to_csv(output_csv, index=False)
        
        print(f"Labeled and normalized features saved to {output_csv}")
        
        # Display labeling summary
        speaker_change_counts = labeled_df['speaker_change'].value_counts()
        print("\nSpeaker Change Label Distribution:")
        print(speaker_change_counts)
        print(f"Percentage of speaker change frames: {speaker_change_counts[1] / len(labeled_df) * 100:.2f}%\n")
        
        # Display top 10 rows of the labeled and normalized CSV
        print("Top 10 rows of the labeled and normalized CSV:")
        print(labeled_df.head(10))  

def main():
    base_path = input("Enter the path to your IEMOCAP dataset: ")
    process_all_sessions(base_path)

if __name__ == "__main__":
    main()


Enter the path to your IEMOCAP dataset:  C:\Users\Admin\SEM5\CPC353\assignment2\IEMOCAP\IEMOCAP\reference


Z-score normalization applied to numerical features.
Labeled and normalized features saved to labeled_features1_session1.csv

Speaker Change Label Distribution:
0    821608
1      3024
Name: speaker_change, dtype: int64
Percentage of speaker change frames: 0.37%

Top 10 rows of the labeled and normalized CSV:
            file_name  frame_index  start_time  end_time        f0    mfcc_1  \
0  Ses01F_impro01.wav            0        0.00      0.02 -0.651306  3.730832   
1  Ses01F_impro01.wav            1        0.01      0.03 -0.651306  2.784090   
2  Ses01F_impro01.wav            2        0.02      0.04 -0.651306  2.535083   
3  Ses01F_impro01.wav            3        0.03      0.05 -0.651306  2.458731   
4  Ses01F_impro01.wav            4        0.04      0.06 -0.651306  2.452730   
5  Ses01F_impro01.wav            5        0.05      0.07 -0.651306  2.108995   
6  Ses01F_impro01.wav            6        0.06      0.08 -0.651306  1.933975   
7  Ses01F_impro01.wav            7        0.07   

In [5]:
def combine_csv_files(output_train, output_val, output_test):
    """
    Combine CSV files into train, validation, and test datasets.
    
    """
    # Initialize empty lists to hold DataFrames for each set
    train_data = []
    validation_data = []
    test_data = []

    # Iterate through labeled CSV files
    for file in os.listdir('.'):
        if file.startswith('labeled_features1_session') and file.endswith('.csv'):
            # Extract session number from file name
            session_num = int(file.split('session')[-1].split('.')[0])
            
            # Read the CSV file
            df = pd.read_csv(file)
            
            # Append to the appropriate dataset based on session number
            if session_num in [1, 2, 3]:
                train_data.append(df)
            elif session_num == 4:
                validation_data.append(df)
            elif session_num == 5:
                test_data.append(df)

    # Combine all DataFrames for each set
    train_df = pd.concat(train_data, ignore_index=True)
    validation_df = pd.concat(validation_data, ignore_index=True)
    test_df = pd.concat(test_data, ignore_index=True)

    # Save the combined DataFrames to CSV files
    train_df.to_csv(output_train, index=False)
    validation_df.to_csv(output_val, index=False)
    test_df.to_csv(output_test, index=False)

    print(f"Training data saved to {output_train} ({len(train_df)} rows)")
    print(f"Validation data saved to {output_val} ({len(validation_df)} rows)")
    print(f"Test data saved to {output_test} ({len(test_df)} rows)")
    
    # Show the first 10 rows of each dataset
    print("\nTop 10 rows of the training dataset:")
    print(train_df.head(10))
    print("\nTop 10 rows of the validation dataset:")
    print(validation_df.head(10))
    print("\nTop 10 rows of the test dataset:")
    print(test_df.head(10))

# Main function
def main():
    # Define output filenames
    output_train = "train_data1.csv"
    output_val = "validation_data1.csv"
    output_test = "test_data1.csv"
    
    # Combine CSV files into train, validation, and test datasets
    combine_csv_files(output_train, output_val, output_test)

if __name__ == "__main__":
    main()


Training data saved to train_data1.csv (2452302 rows)
Validation data saved to validation_data1.csv (834063 rows)
Test data saved to test_data1.csv (879281 rows)

Top 10 rows of the training dataset:
            file_name  frame_index  start_time  end_time        f0    mfcc_1  \
0  Ses01F_impro01.wav            0        0.00      0.02 -0.651306  3.730832   
1  Ses01F_impro01.wav            1        0.01      0.03 -0.651306  2.784090   
2  Ses01F_impro01.wav            2        0.02      0.04 -0.651306  2.535083   
3  Ses01F_impro01.wav            3        0.03      0.05 -0.651306  2.458731   
4  Ses01F_impro01.wav            4        0.04      0.06 -0.651306  2.452730   
5  Ses01F_impro01.wav            5        0.05      0.07 -0.651306  2.108995   
6  Ses01F_impro01.wav            6        0.06      0.08 -0.651306  1.933975   
7  Ses01F_impro01.wav            7        0.07      0.09 -0.651306  1.808417   
8  Ses01F_impro01.wav            8        0.08      0.10 -0.651306  1.746722   


## Experiment 2
**frame_length/sliding window:** 80ms

**frame_shift/overlap:** 10ms 

In [6]:
import os
import pandas as pd
from sklearn.preprocessing import StandardScaler

def read_rttm_file(rttm_path):
    """
    Read RTTM file and return a list of speaker segments
    """
    speaker_segments = []
    with open(rttm_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if parts[0] == 'SPEAKER':
                speaker_segments.append({
                    'file': parts[1],
                    'time_start': float(parts[3]),
                    'duration': float(parts[4]),
                    'end_time': float(parts[3]) + float(parts[4]),
                })
    return speaker_segments

def label_speaker_changes(features_df, base_path):
    """
    Label speaker changes based on matching RTTM files
    """
    # Create a copy of the DataFrame to avoid modifying the original
    labeled_df = features_df.copy()
    
    # Add speaker change label column
    labeled_df['speaker_change'] = 0
    
    # Find unique wav files
    unique_wav_files = features_df['file_name'].unique()
    
    for wav_file in unique_wav_files:
        # Remove .wav extension and find matching RTTM
        base_filename = os.path.splitext(wav_file)[0]
        
        # Search for RTTM file in the entire base path
        matching_rttm_files = []
        for root, dirs, files in os.walk(base_path):
            matching_rttm_files.extend([
                os.path.join(root, f) for f in files 
                if base_filename in f and f.endswith('.rttm')
            ])
        
        if not matching_rttm_files:
            print(f"No RTTM file found for {wav_file}")
            continue
        
        # Use the first matching RTTM file
        rttm_path = matching_rttm_files[0]
        
        # Read speaker segments from RTTM file
        speaker_segments = read_rttm_file(rttm_path)
        
        # Label speaker changes
        for segment in speaker_segments:
            # Find rows within the speaker segment
            mask = (
               (labeled_df['file_name'] == wav_file) & 
               (labeled_df['start_time'] < segment['time_start']) & 
               (labeled_df['end_time'] >= segment['time_start'])
           )
            
            labeled_df.loc[mask, 'speaker_change'] = 1
    
    return labeled_df

def normalize_features_zscore(labeled_df):
    """
    Normalize numerical features using Z-score normalization
    """
    # Select numerical feature columns
    feature_columns = [col for col in labeled_df.columns if col.startswith('mfcc') or col == 'f0']
    
    # Initialize StandardScaler
    scaler = StandardScaler()
    
    # Apply Z-score normalization
    labeled_df[feature_columns] = scaler.fit_transform(labeled_df[feature_columns])
    
    print("Z-score normalization applied to numerical features.")
    return labeled_df

def process_all_sessions(base_path):
    """
    Process all sessions, label speaker changes, and normalize features
    """
    # Find feature CSV files
    feature_files = [f for f in os.listdir('.') if f.startswith('features2_') and f.endswith('.csv')]
    
    for features_csv in feature_files:
        # Load features
        features_df = pd.read_csv(features_csv)
        
        # Label speaker changes
        labeled_df = label_speaker_changes(features_df, base_path)
        
        # Normalize numerical features
        labeled_df = normalize_features_zscore(labeled_df)
        
        # Save labeled and normalized features
        output_csv = f'labeled_{features_csv}'
        labeled_df.to_csv(output_csv, index=False)
        
        print(f"Labeled and normalized features saved to {output_csv}")
        
        # Display labeling summary
        speaker_change_counts = labeled_df['speaker_change'].value_counts()
        print("\nSpeaker Change Label Distribution:")
        print(speaker_change_counts)
        print(f"Percentage of speaker change frames: {speaker_change_counts[1] / len(labeled_df) * 100:.2f}%\n")
        
        # Display top 10 rows of the labeled and normalized CSV
        print("Top 10 rows of the labeled and normalized CSV:")
        print(labeled_df.head(10))  

def main():
    base_path = input("Enter the path to your IEMOCAP dataset: ")
    process_all_sessions(base_path)

if __name__ == "__main__":
    main()


Enter the path to your IEMOCAP dataset:  C:\Users\Admin\SEM5\CPC353\assignment2\IEMOCAP\IEMOCAP\reference


Z-score normalization applied to numerical features.
Labeled and normalized features saved to labeled_features2_session1.csv

Speaker Change Label Distribution:
0    812550
1     12082
Name: speaker_change, dtype: int64
Percentage of speaker change frames: 1.47%

Top 10 rows of the labeled and normalized CSV:
            file_name  frame_index  start_time  end_time        f0    mfcc_1  \
0  Ses01F_impro01.wav            0        0.00      0.08 -0.651306  3.310138   
1  Ses01F_impro01.wav            1        0.01      0.09 -0.651306  3.330899   
2  Ses01F_impro01.wav            2        0.02      0.10 -0.651306  3.123099   
3  Ses01F_impro01.wav            3        0.03      0.11 -0.651306  2.736598   
4  Ses01F_impro01.wav            4        0.04      0.12 -0.651306  2.485984   
5  Ses01F_impro01.wav            5        0.05      0.13 -0.651306  2.307087   
6  Ses01F_impro01.wav            6        0.06      0.14 -0.651306  2.112641   
7  Ses01F_impro01.wav            7        0.07   

In [7]:
def combine_csv_files(output_train, output_val, output_test):
    """
    Combine CSV files into train, validation, and test datasets.
    
    Parameters:
        output_train: Name of the output training CSV file.
        output_val: Name of the output validation CSV file.
        output_test: Name of the output test CSV file.
    """
    # Initialize empty lists to hold DataFrames for each set
    train_data = []
    validation_data = []
    test_data = []

    # Iterate through labeled CSV files
    for file in os.listdir('.'):
        if file.startswith('labeled_features2_session') and file.endswith('.csv'):
            # Extract session number from file name
            session_num = int(file.split('session')[-1].split('.')[0])
            
            # Read the CSV file
            df = pd.read_csv(file)
            
            # Append to the appropriate dataset based on session number
            if session_num in [1, 2, 3]:
                train_data.append(df)
            elif session_num == 4:
                validation_data.append(df)
            elif session_num == 5:
                test_data.append(df)

    # Combine all DataFrames for each set
    train_df = pd.concat(train_data, ignore_index=True)
    validation_df = pd.concat(validation_data, ignore_index=True)
    test_df = pd.concat(test_data, ignore_index=True)

    # Save the combined DataFrames to CSV files
    train_df.to_csv(output_train, index=False)
    validation_df.to_csv(output_val, index=False)
    test_df.to_csv(output_test, index=False)

    print(f"Training data saved to {output_train} ({len(train_df)} rows)")
    print(f"Validation data saved to {output_val} ({len(validation_df)} rows)")
    print(f"Test data saved to {output_test} ({len(test_df)} rows)")

        # Show the first 10 rows of each dataset
    print("\nTop 10 rows of the training dataset:")
    print(train_df.head(10))
    print("\nTop 10 rows of the validation dataset:")
    print(validation_df.head(10))
    print("\nTop 10 rows of the test dataset:")
    print(test_df.head(10))

# Main function
def main():
    # Define output filenames
    output_train = "train_data2.csv"
    output_val = "validation_data2.csv"
    output_test = "test_data2.csv"
    
    # Combine CSV files into train, validation, and test datasets
    combine_csv_files(output_train, output_val, output_test)

if __name__ == "__main__":
    main()


Training data saved to train_data2.csv (2452302 rows)
Validation data saved to validation_data2.csv (834063 rows)
Test data saved to test_data2.csv (879281 rows)

Top 10 rows of the training dataset:
            file_name  frame_index  start_time  end_time        f0    mfcc_1  \
0  Ses01F_impro01.wav            0        0.00      0.08 -0.651306  3.310138   
1  Ses01F_impro01.wav            1        0.01      0.09 -0.651306  3.330899   
2  Ses01F_impro01.wav            2        0.02      0.10 -0.651306  3.123099   
3  Ses01F_impro01.wav            3        0.03      0.11 -0.651306  2.736598   
4  Ses01F_impro01.wav            4        0.04      0.12 -0.651306  2.485984   
5  Ses01F_impro01.wav            5        0.05      0.13 -0.651306  2.307087   
6  Ses01F_impro01.wav            6        0.06      0.14 -0.651306  2.112641   
7  Ses01F_impro01.wav            7        0.07      0.15 -0.651306  1.947617   
8  Ses01F_impro01.wav            8        0.08      0.16 -0.651306  1.848644   
